# XML Feature Extraction - 2D and 3D

#### Environment with numpy==1.20.3 e pylidc==0.2.2 (new_env kernel)

In [1]:
import os
import six
import pickle
import pandas as pd
import SimpleITK as sitk
from functools import reduce
import numpy as np
from scipy.stats import mode
from skimage.measure import find_contours
from skimage import io
import pydicom
import pylidc as pl
from pylidc.utils import consensus
import matplotlib.pyplot as plt

In [11]:
import sys
print(sys.executable)


C:\Users\user\anaconda3\envs\new_env\python.exe


#### std_limit = 0.5

In [2]:
import os
import six
import pickle
import pandas as pd
import SimpleITK as sitk
from functools import reduce
import numpy as np
from scipy.stats import mode
from skimage.measure import find_contours
from skimage import io
import pydicom
import pylidc as pl
from pylidc.utils import consensus
import matplotlib.pyplot as plt

In [4]:
import os
import numpy as np
import pandas as pd
import pylidc as pl
from pylidc.utils import consensus

# Paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_0.5"
external_drive_path = "C:/Users/user/Desktop/PI_project"  # Path to save CSV

# ✅ Ensure output folder exists
os.makedirs(path, exist_ok=True)

# Initialize patient folders
patient_folders = sorted(os.listdir(input_directory))
nodule_id = 0

# Dictionary to collect nodule data
nodule_data = {
    'Nodule_id': [],
    'Patient_id': [],
    'Subtlety': [],
    'Internalstructure': [],
    'Calcification': [],
    'Sphericity': [],
    'Margin': [],
    'Lobulation': [],
    'Spiculation': [],
    'Texture': [],
    'Malignancy': []
}

malignancy_names = {
    1: "1-Highly Unlikely",
    2: "2-Moderately Unlikely",
    3: "3-Indeterminate",
    4: "4-Moderately Suspicious",
    5: "5-Highly Suspicious"
}

std_limit = 0.5

# Process each patient
for patient_folder in patient_folders:
    patient_folder_path = os.path.join(input_directory, patient_folder)
    patient_id = os.path.basename(patient_folder_path)
    print(f"Processing patient: {patient_id}")
    
    patient_scans = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id, pl.Scan.annotations.any()).all()
    
    for scan in patient_scans:
        nods = scan.cluster_annotations()
        
        for anns in nods:
            nodule_id += 1
            nodule_characteristics = []

            # 50% consensus mask
            cmask, cbbox, masks = consensus(anns, clevel=0.5, pad=[(20, 20), (20, 20), (0, 0)])

            for i, mask in enumerate(masks):
                characteristics = anns[i].feature_vals()
                nodule_characteristics.append(characteristics)

            if nodule_characteristics:
                std_deviations = [np.std(characteristic) for characteristic in zip(*nodule_characteristics)]
                
                # Skip if any feature has std > limit
                if any(std > std_limit for std in std_deviations):
                    print(f"Skipped Nodule ID {nodule_id} due to high std.")
                    continue

                # ✅ Save cmask
                filename = os.path.join(path, f"{nodule_id}_cmask.npy")
                np.save(filename, cmask)

                # Optional volume save — uncomment if needed and storage allows
                # volume = anns[0].scan.to_volume()
                # filename = os.path.join(path, f"{nodule_id}_volume.npy")
                # np.save(filename, volume)

                # Extract characteristics (from first annotation)
                feature_values = {
                    'Nodule_id': nodule_id,
                    'Patient_id': patient_id,
                    'Subtlety': nodule_characteristics[0][0],
                    'Internalstructure': nodule_characteristics[0][1],
                    'Calcification': nodule_characteristics[0][2],
                    'Sphericity': nodule_characteristics[0][3],
                    'Margin': nodule_characteristics[0][4],
                    'Lobulation': nodule_characteristics[0][5],
                    'Spiculation': nodule_characteristics[0][6],
                    'Texture': nodule_characteristics[0][7],
                    'Malignancy': malignancy_names.get(nodule_characteristics[0][8])
                }

                for key, value in feature_values.items():
                    nodule_data[key].append(value)

# Save features to CSV
features = pd.DataFrame(nodule_data)
csv_filename = os.path.join(external_drive_path, 'features_pylidc_0.5.csv')
features.to_csv(csv_filename, index=False)

print("✅ Feature extraction complete and CSV saved.")


Processing patient: LICENSE
Processing patient: LIDC-IDRI-0001
Skipped Nodule ID 1 due to high std.
Processing patient: LIDC-IDRI-0002
Skipped Nodule ID 2 due to high std.
Processing patient: LIDC-IDRI-0003
Skipped Nodule ID 4 due to high std.
Skipped Nodule ID 5 due to high std.
Skipped Nodule ID 6 due to high std.
Processing patient: LIDC-IDRI-0004
Skipped Nodule ID 7 due to high std.
Processing patient: LIDC-IDRI-0005
Skipped Nodule ID 8 due to high std.
Skipped Nodule ID 9 due to high std.
Processing patient: LIDC-IDRI-0006
Skipped Nodule ID 12 due to high std.
Skipped Nodule ID 14 due to high std.
Processing patient: LIDC-IDRI-0007
Skipped Nodule ID 15 due to high std.
Processing patient: LIDC-IDRI-0008
Skipped Nodule ID 17 due to high std.
Skipped Nodule ID 18 due to high std.
Processing patient: LIDC-IDRI-0009
Processing patient: LIDC-IDRI-0010
Skipped Nodule ID 22 due to high std.
Skipped Nodule ID 23 due to high std.
Processing patient: LIDC-IDRI-0011
Skipped Nodule ID 24 due 

#### std_limit = 1.0

In [7]:
import os
import numpy as np
import pandas as pd
import pylidc as pl
from pylidc.utils import consensus

# Paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_1.0"
external_drive_path = "C:/Users/user/Desktop/PI_project"  # Path to save CSV

# ✅ Ensure output folder exists
os.makedirs(path, exist_ok=True)

# Initialize patient folders
patient_folders = sorted(os.listdir(input_directory))
nodule_id = 0

# Dictionary to collect nodule data
nodule_data = {
    'Nodule_id': [],
    'Patient_id': [],
    'Subtlety': [],
    'Internalstructure': [],
    'Calcification': [],
    'Sphericity': [],
    'Margin': [],
    'Lobulation': [],
    'Spiculation': [],
    'Texture': [],
    'Malignancy': []
}

malignancy_names = {
    1: "1-Highly Unlikely",
    2: "2-Moderately Unlikely",
    3: "3-Indeterminate",
    4: "4-Moderately Suspicious",
    5: "5-Highly Suspicious"
}

std_limit = 1.0

# Process each patient
for patient_folder in patient_folders:
    patient_folder_path = os.path.join(input_directory, patient_folder)
    patient_id = os.path.basename(patient_folder_path)
    print(f"Processing patient: {patient_id}")
    
    patient_scans = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id, pl.Scan.annotations.any()).all()
    
    for scan in patient_scans:
        nods = scan.cluster_annotations()
        
        for anns in nods:
            nodule_id += 1
            nodule_characteristics = []

            # 50% consensus mask
            cmask, cbbox, masks = consensus(anns, clevel=0.5, pad=[(20, 20), (20, 20), (0, 0)])

            for i, mask in enumerate(masks):
                characteristics = anns[i].feature_vals()
                nodule_characteristics.append(characteristics)

            if nodule_characteristics:
                std_deviations = [np.std(characteristic) for characteristic in zip(*nodule_characteristics)]
                
                # Skip if any feature has std > limit
                if any(std > std_limit for std in std_deviations):
                    print(f"Skipped Nodule ID {nodule_id} due to high std.")
                    continue

                # ✅ Save cmask
                filename = os.path.join(path, f"{nodule_id}_cmask.npy")
                np.save(filename, cmask)

                # Optional volume save — uncomment if needed and storage allows
                # volume = anns[0].scan.to_volume()
                # filename = os.path.join(path, f"{nodule_id}_volume.npy")
                # np.save(filename, volume)

                # Extract characteristics (from first annotation)
                feature_values = {
                    'Nodule_id': nodule_id,
                    'Patient_id': patient_id,
                    'Subtlety': nodule_characteristics[0][0],
                    'Internalstructure': nodule_characteristics[0][1],
                    'Calcification': nodule_characteristics[0][2],
                    'Sphericity': nodule_characteristics[0][3],
                    'Margin': nodule_characteristics[0][4],
                    'Lobulation': nodule_characteristics[0][5],
                    'Spiculation': nodule_characteristics[0][6],
                    'Texture': nodule_characteristics[0][7],
                    'Malignancy': malignancy_names.get(nodule_characteristics[0][8])
                }

                for key, value in feature_values.items():
                    nodule_data[key].append(value)

# Save features to CSV
features = pd.DataFrame(nodule_data)
csv_filename = os.path.join(external_drive_path, 'features_pylidc_1.0.csv')
features.to_csv(csv_filename, index=False)

print("✅ Feature extraction complete and CSV saved.")


Processing patient: LICENSE
Processing patient: LIDC-IDRI-0001
Skipped Nodule ID 1 due to high std.
Processing patient: LIDC-IDRI-0002
Processing patient: LIDC-IDRI-0003
Skipped Nodule ID 4 due to high std.
Skipped Nodule ID 5 due to high std.
Skipped Nodule ID 6 due to high std.
Processing patient: LIDC-IDRI-0004
Skipped Nodule ID 7 due to high std.
Processing patient: LIDC-IDRI-0005
Skipped Nodule ID 8 due to high std.
Processing patient: LIDC-IDRI-0006
Skipped Nodule ID 14 due to high std.
Processing patient: LIDC-IDRI-0007
Skipped Nodule ID 15 due to high std.
Processing patient: LIDC-IDRI-0008
Skipped Nodule ID 17 due to high std.
Skipped Nodule ID 18 due to high std.
Processing patient: LIDC-IDRI-0009
Processing patient: LIDC-IDRI-0010
Skipped Nodule ID 23 due to high std.
Processing patient: LIDC-IDRI-0011
Skipped Nodule ID 25 due to high std.
Skipped Nodule ID 29 due to high std.
Skipped Nodule ID 33 due to high std.
Processing patient: LIDC-IDRI-0012
Skipped Nodule ID 35 due t

#### std_limit = 1.5

In [8]:
import os
import numpy as np
import pandas as pd
import pylidc as pl
from pylidc.utils import consensus

# Paths
input_directory = "C:/Users/user/Desktop/PI_project/images/LIDC-IDRI"
params_file = "C:/Users/user/Desktop/PI_project/radiomics_config/Params.yaml"
path = "C:/Users/user/Desktop/PI_project/std_limit_1.5"
external_drive_path = "C:/Users/user/Desktop/PI_project"  # Path to save CSV

# ✅ Ensure output folder exists
os.makedirs(path, exist_ok=True)

# Initialize patient folders
patient_folders = sorted(os.listdir(input_directory))
nodule_id = 0

# Dictionary to collect nodule data
nodule_data = {
    'Nodule_id': [],
    'Patient_id': [],
    'Subtlety': [],
    'Internalstructure': [],
    'Calcification': [],
    'Sphericity': [],
    'Margin': [],
    'Lobulation': [],
    'Spiculation': [],
    'Texture': [],
    'Malignancy': []
}

malignancy_names = {
    1: "1-Highly Unlikely",
    2: "2-Moderately Unlikely",
    3: "3-Indeterminate",
    4: "4-Moderately Suspicious",
    5: "5-Highly Suspicious"
}

std_limit = 1.5

# Process each patient
for patient_folder in patient_folders:
    patient_folder_path = os.path.join(input_directory, patient_folder)
    patient_id = os.path.basename(patient_folder_path)
    print(f"Processing patient: {patient_id}")
    
    patient_scans = pl.query(pl.Scan).filter(pl.Scan.patient_id == patient_id, pl.Scan.annotations.any()).all()
    
    for scan in patient_scans:
        nods = scan.cluster_annotations()
        
        for anns in nods:
            nodule_id += 1
            nodule_characteristics = []

            # 50% consensus mask
            cmask, cbbox, masks = consensus(anns, clevel=0.5, pad=[(20, 20), (20, 20), (0, 0)])

            for i, mask in enumerate(masks):
                characteristics = anns[i].feature_vals()
                nodule_characteristics.append(characteristics)

            if nodule_characteristics:
                std_deviations = [np.std(characteristic) for characteristic in zip(*nodule_characteristics)]
                
                # Skip if any feature has std > limit
                if any(std > std_limit for std in std_deviations):
                    print(f"Skipped Nodule ID {nodule_id} due to high std.")
                    continue

                # ✅ Save cmask
                filename = os.path.join(path, f"{nodule_id}_cmask.npy")
                np.save(filename, cmask)

                # Optional volume save — uncomment if needed and storage allows
                # volume = anns[0].scan.to_volume()
                # filename = os.path.join(path, f"{nodule_id}_volume.npy")
                # np.save(filename, volume)

                # Extract characteristics (from first annotation)
                feature_values = {
                    'Nodule_id': nodule_id,
                    'Patient_id': patient_id,
                    'Subtlety': nodule_characteristics[0][0],
                    'Internalstructure': nodule_characteristics[0][1],
                    'Calcification': nodule_characteristics[0][2],
                    'Sphericity': nodule_characteristics[0][3],
                    'Margin': nodule_characteristics[0][4],
                    'Lobulation': nodule_characteristics[0][5],
                    'Spiculation': nodule_characteristics[0][6],
                    'Texture': nodule_characteristics[0][7],
                    'Malignancy': malignancy_names.get(nodule_characteristics[0][8])
                }

                for key, value in feature_values.items():
                    nodule_data[key].append(value)

# Save features to CSV
features = pd.DataFrame(nodule_data)
csv_filename = os.path.join(external_drive_path, 'features_pylidc_1.5.csv')
features.to_csv(csv_filename, index=False)

print("✅ Feature extraction complete and CSV saved.")


Processing patient: LICENSE
Processing patient: LIDC-IDRI-0001
Processing patient: LIDC-IDRI-0002
Processing patient: LIDC-IDRI-0003
Processing patient: LIDC-IDRI-0004
Processing patient: LIDC-IDRI-0005
Processing patient: LIDC-IDRI-0006
Processing patient: LIDC-IDRI-0007
Processing patient: LIDC-IDRI-0008
Skipped Nodule ID 17 due to high std.
Processing patient: LIDC-IDRI-0009
Processing patient: LIDC-IDRI-0010
Processing patient: LIDC-IDRI-0011
Processing patient: LIDC-IDRI-0012
Processing patient: LIDC-IDRI-0013
Processing patient: LIDC-IDRI-0014
Processing patient: LIDC-IDRI-0015
Processing patient: LIDC-IDRI-0016
Skipped Nodule ID 54 due to high std.
Skipped Nodule ID 55 due to high std.
Processing patient: LIDC-IDRI-0017
Processing patient: LIDC-IDRI-0018
Skipped Nodule ID 61 due to high std.
Processing patient: LIDC-IDRI-0019
Processing patient: LIDC-IDRI-0020
Processing patient: LIDC-IDRI-0021
Processing patient: LIDC-IDRI-0022
Processing patient: LIDC-IDRI-0023
Processing pati

LIDC-IDRI-0943
LIDC-IDRI-0944
Nodule ID 2486
LIDC-IDRI-0945
LIDC-IDRI-0946
LIDC-IDRI-0947
LIDC-IDRI-0949
Nodule ID 2496
LIDC-IDRI-0950
LIDC-IDRI-0951
LIDC-IDRI-0953
Nodule ID 2506
LIDC-IDRI-0955
LIDC-IDRI-0956
Nodule ID 2509
Nodule ID 2510
LIDC-IDRI-0957
LIDC-IDRI-0958
LIDC-IDRI-0959
LIDC-IDRI-0961
Nodule ID 2517
Nodule ID 2523
LIDC-IDRI-0962
LIDC-IDRI-0963
LIDC-IDRI-0965
LIDC-IDRI-0966
LIDC-IDRI-0968
Nodule ID 2533
LIDC-IDRI-0969
Nodule ID 2536
LIDC-IDRI-0971
LIDC-IDRI-0972
LIDC-IDRI-0973
LIDC-IDRI-0974
LIDC-IDRI-0976
LIDC-IDRI-0977
LIDC-IDRI-0978
LIDC-IDRI-0980
Nodule ID 2556
LIDC-IDRI-0981
LIDC-IDRI-0982
LIDC-IDRI-0983
LIDC-IDRI-0984
Nodule ID 2564
LIDC-IDRI-0985
LIDC-IDRI-0986
LIDC-IDRI-0987
Nodule ID 2571
LIDC-IDRI-0989
LIDC-IDRI-0990
LIDC-IDRI-0991
LIDC-IDRI-0993
LIDC-IDRI-0994
Nodule ID 2581
Nodule ID 2582
LIDC-IDRI-0996
LIDC-IDRI-0997
LIDC-IDRI-0998
Nodule ID 2597
LIDC-IDRI-0999
LIDC-IDRI-1000
Nodule ID 2606
Nodule ID 2608
Nodule ID 2609
LIDC-IDRI-1001
LIDC-IDRI-1002
LIDC-IDRI-